# 07. Solver Framework Design

- Parent issue: `#23`
- Status: `active`
- Summary: Implement the `Solver`/`Verifier` Protocol, `CategoryRouter`, and `BitManipulationSolver`. Produces `src/inference/solver.py`, `src/solvers/bit_manipulation.py`, `configs/solver_routing.yaml`, and `docs/architecture/SOLVER_DESIGN.md`.

## Audience and Why It Matters

Agents building solvers, reviewers checking extensibility, and stakeholders evaluating the project’s differentiation strategy.


## Decision / Hypothesis

Use a category-aware `solve` and `verify` contract with explicit confidence and metadata so solvers and LLM teachers can coexist.


## Environment and Reproduction

- Python: 3.11+, `pyyaml` required (`pip install pyyaml`)
- Run from repo root: `jupyter notebook notebooks/07_solver_framework_design.ipynb`
- No GPU required — this notebook documents interface decisions and runs a worked example
- Consumes: category list from `#17`/`#22`; uses fallback categories if not available
- Companion registry entry: `docs/execution/NOTEBOOKS.md`

In [ ]:
from pathlib import Path
import platform

REPO_ROOT = Path.cwd()

print(f"Repo root: {REPO_ROOT}")
print(f"Python platform: {platform.platform()}")


In [ ]:
# Device detection: prefers CUDA, falls back to CPU gracefully.
import sys
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir() and (_candidate / 'notebooks').is_dir():
        REPO_ROOT = _candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.device import log_device_info
device = log_device_info()


## Method and Outputs

### Method

1. Show `SolverOutput`, `Solver`, `Verifier` Protocol contracts from `src/inference/solver.py`.
2. Instantiate `CategoryRouter`; register `BitManipulationSolver`.
3. Demonstrate `route(prompt, category)` with a worked example.
4. Show `FailureMode` values and when each fires.
5. Golden-set regression: if `#18` golden set is available, verify solver adds no regression.

### Outputs produced

- `src/inference/solver.py` — Protocol definitions, CategoryRouter, FailureMode
- `src/solvers/bit_manipulation.py` — first concrete solver (#9 / #26)
- `configs/solver_routing.yaml` — category → confidence threshold mapping
- `docs/architecture/SOLVER_DESIGN.md` — design rationale and risk mitigations

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd()
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from src.inference.solver import (
    CategoryRouter, FailureMode, Solver, SolverOutput, Verifier
)
from src.solvers.bit_manipulation import BitManipulationSolver

# Category list: from #17/#22, or fallback
CATEGORIES = ["bit_manipulation", "math", "code", "science"]

print("Solver framework loaded.")
print(f"Categories: {CATEGORIES}")
print(f"FailureMode values: {[fm.value for fm in FailureMode]}")

In [ ]:
# Show SolverOutput shape
output = SolverOutput.make(answer=r"\boxed{42}", confidence=0.95, metadata={"rule": "lshift_2"})
print("SolverOutput shape:")
for k, v in output.items():
    print(f"  {k}: {v!r}")

# Show Solver Protocol structural check
solver = BitManipulationSolver(confidence_threshold=0.7)
assert isinstance(solver, Solver), "BitManipulationSolver must satisfy Solver Protocol"
print("\nBitManipulationSolver implements Solver Protocol: OK")

In [ ]:
# Worked example: CategoryRouter with BitManipulationSolver
router = CategoryRouter()
router.register("bit_manipulation", BitManipulationSolver(confidence_threshold=0.7))
router._thresholds["bit_manipulation"] = 0.7

WORKED_PROMPT = (
    "Find the rule:\n"
    "1 → 4\n"
    "2 → 8\n"
    "3 → 12\n"
    "What is 5 → ?"
)

result = router.route(WORKED_PROMPT, "bit_manipulation")
print("router.route(prompt, 'bit_manipulation'):")
print(f"  answer:     {result['answer']}")
print(f"  confidence: {result['confidence']:.2f}")
print(f"  metadata:   {result['metadata']}")
print()

# Show LLM fallback path
result_unregistered = router.route("some math problem", "math")
print("router.route(prompt, 'math')  [no solver registered]:")
print(f"  answer:       {result_unregistered['answer']}")
print(f"  failure_mode: {result_unregistered['metadata'].get('failure_mode')}")

## Results / Open Risks

**Artifacts produced:**
- `src/inference/solver.py` — `Solver`/`Verifier` Protocols, `SolverOutput`, `FailureMode`, `CategoryRouter`
- `src/solvers/bit_manipulation.py` — first category solver (#9 / #26)
- `configs/solver_routing.yaml` — routing config with per-category thresholds
- `docs/architecture/SOLVER_DESIGN.md` — design rationale

**Open risks:**
- How many categories does `#22` discover? Next solver (#26) category to be decided from error-slice taxonomy.
- If SFT shows no per-category loss divergence, collapse to single adapter with category-aware decode only.
- `Verifier` Protocol is structural — Branch 3 (`#24`) must implement a concrete verifier before the quality filter can use `verify()`.

## Sources

            - [Architecture doc](docs/architecture/ARCHITECTURE.md)
- [Adversarial review](docs/analysis/ADVERSARIAL_REVIEW.md)
- [Tong Hui Kang repo](https://github.com/tonghuikang/nemotron)
